# Task C. Cryptographic Hash Functions (10 marks)

In this task, we aim to let students develop a solid understanding of the construction and the properties of cryptographic hash functions, as well as experience the use of modern cryptographic hash functions such as SHA512. Refer to https://cryptography.io/en/latest/hazmat/primitives/cryptographic-hashes/ for detailed documentation for the SHA-2 family and MD5.

In the following source code, we have implemented a **simple hash function** and the **birthday attack** targeting this hash function. 
* The simple hash function is a truncated version of MD5, which uses part of MD5's digest as the hash values. 
* The birthday attack is implemented by uniformly randomly generating a pair of messages from a message space without replacement, and checking if their hash values collide. The number of examined hash values before the first collision is recorded and returned, together with the two collided messages. We use the output file from Task B.3 named “\<YourStudentID\>_Task-B-3_plain-text.txt” as the message set and each line in the file is regarded as a message.
  
We have demonstrated examples of using the hash function and performing the birthday attack once.You are required to read and understand the source code to complete the followed tasks.

In [1]:
from cryptography.hazmat.primitives import hashes
import random
import math
import binascii

#=====================================================
# Please do NOT modify the following code, but you are more than welcome to understand the code in detail

# A hash function class defined for Task C.
# This is trncated version of MD5.
class TruncatedMD5:

    # Use the first 8 bits as the default
    def __init__(self, n_bits=8):
        self.n_bits = n_bits

    def get_hash_value(self, message=None):
        
        if(None==message or 0==len(message)):
            sys.exit("No message is given!")

        # Encode the message to bytes
        message_bytes = bytes(message, "utf-8")
        
        digest = hashes.Hash(hashes.MD5())
        digest.update(message_bytes)
        hash_value_byte = digest.finalize()

        # Decode the bytes to hexadecimal representation, i.e., each half byte (or 4 bits) will be decoded as a hexadecimal number (1, 2, ..., 10, a, b, c, d, e, f).
        hash_value_hex = binascii.hexlify(hash_value_byte).decode("utf-8")

        # Return the first n_bits by truncation
        return hash_value_hex[:self.n_bits//4]


# Implement the birthday attack on a hash function and a specific set of messages
def birthday_attack(message_space, hash_function):
    
    tried_hash_values = set()
    tried_message_indices = {}
    collision_pair_1 = -1
    collision_pair_2 = -1
    collision_hash_value = -1
    
    random_index_samples = random.sample(range(len(message_space)), len(message_space)); # Random sampling without replacement
    
    for i in random_index_samples:
        message = message_space[i]
        hash_value = hash_function.get_hash_value(message)

        
        if hash_value not in tried_hash_values: # No collision so far
            tried_hash_values.add(hash_value)
            tried_message_indices[hash_value] = i
        elif message != message_space[tried_message_indices[hash_value]]: # A collision is found
            collision_pair_1 = i
            collision_pair_2 = tried_message_indices[hash_value]
            collision_hash_value = hash_value
            return len(tried_hash_values), collision_pair_1, collision_pair_2, collision_hash_value
    
    # No collision is found
    print("No collision is found!")
    return -1, -1, -1, -1

#=====================================================
# Examples of using the defined hash function
hash_function = TruncatedMD5(8)

example_message = "Hellow World, COMP2300/6300!"
print("Example message is: ", example_message)
print("Hash value of example message is: ", hash_function.get_hash_value(example_message))

example_message_variant = "Hell0w World, COMP2300/6300!"
print("Example message variant is: ", example_message_variant)
print("Hash value of example message variant is: ", hash_function.get_hash_value(example_message_variant))
print("")

#=====================================================
# An example birthday attack on the defined hash function.

# We use each line of the plaintext in Task B.3 as a message. Then, the whole text can be regarded as a set of messages.
# Note that you need to replace the plain text file name (i.e., to replace <YourStudentID> with your own ID) to execute the program correctly
message_list  = None
with open("<YourStudentID>_Task-B-3_plain-text.txt") as file:
    # This is to filter out the empty lines
    message_list = [line.strip() for line in file if line.strip()]

# Show the number of messages
print("Count of messages: ", len(message_list))

# Perform the birthday_attack to the hash function once.
count, cp1, cp2, digest = birthday_attack(message_list, hash_function)

print("The number of examined hash values (NOT the number of examined messages) before the first collision: \n%i" %count)
print("Collided message 1: ", cp1)
print("Collided message 1: ", message_list[cp1])
print("Collided message 2: ", cp2)
print("Collided message 2: ", message_list[cp2])
print("collided digest: ", digest)
print("The count before collision: ", count)
print("")

Example message is:  Hellow World, COMP2300/6300!
Hash value of example message is:  d8
Example message variant is:  Hell0w World, COMP2300/6300!
Hash value of example message variant is:  64

Count of messages:  305
The number of examined hash values (NOT the number of examined messages) before the first collision: 
16
Collided message 1:  9
Collided message 1:  disclaims all liability for damages resulting from their use to the
Collided message 2:  114
Collided message 2:  a. reproduce and Share the Licensed Material, in whole or
collided digest:  b1
The count before collision:  16



### Task C.1 (4 marks)

Use the **same setting** as the example (i.e., the same hash function and the same message set from the same file "\<YourStudentID\>_Task-B-3_plain-text.txt") demonstrated above.
* You are required to perform to the birthday attack multiple times, and calculate the average number of examined hash values before the first collision.
* Compare the empirical average number of examined hash values before the first collision with its theoretical value (i.e., $\sqrt{\pi/2*n}$, where $n$ is the number of possible hash values) to check if the difference is significant.
  
With writing the source code, you should answer the following questions.

**Question C.1.1 (1 mark)**: What are the empirical average number of examined hash values before the first collision and the standard deviation? You can decide how many rounds you wish to repeat the execution of birthday attachk, but the result should be statistically significant. You need to write souce code to answer this question. Please provide in the answer the rounds you run the birthday attacks, the average of numbers of the examined hash values before the first collision, and the standard deviation.

**Answer**: 

**Question C.1.2 (1 mark)**: Comparing with the theoretical value of the average number of examined hash values before the first collision, please describe and explain the difference between the theoretical value and the empirical one obtained in **Question C.1.1**. Please provide in the answer the theoretical value, whether the diference is significant or not, and the explanation.

**Answer**: 

**Question C.1.3 (1 mark)**: If you construct the TruncatedMD5 has function with the parameter **n_bit=16**, you can get longer hash codes (two bytes or 4 hexadecimal number). In such a case, please describe and explain the difference between the theoretical and empirical values.

**Answer**: 

**Task C.1 Souce Code (1 mark)**

In [3]:
# TODO: Your code to achieve the subtasks above



## Task C.2 (3 marks)

Through understanding the construction of the hash function TruncatedMD5, you are requested to forge a message for a given message, i.e., to identify a second preimage. 
* The given message is one in the message list, indexed by the value $\text{<YourStudentID>}~ (\text{mod} ~97)$, i.e., the last two digits of your student ID. The message list is the same one constructed from the file "\<YourStudentID\>_Task-B-3_plain-text.txt", as used in the example source code.
* **The change should be minor with substitution of one place**, and the forged version should still produce the same hash value as the given authentic message. Specifically, the change can be implemented by one of the following substitutions:
    * For an upper-case letter in the message, it can be replaced by one of any other upper-case letters;
    * For a lower-case letter in the message, it can be replaced by one of any other lower-case letters;
    * For a number in the message, it can be replaced by one of any other numbers.
* To find out a forged message, you can generate a message by changing one placeof the given message. If the generated message produces the same digest, it is a successfully forged message.
* Throughout this task, the parameter n_bits for TruncatedMD5 is n_bits=8, i.e., the same as that in the given example.

**Question C.2.1 (0.5 mark)**: What is the message you choose according to your student ID?

**Answer**:

**Question C.2.2 (0.5 mark)**: What is the digest of the message of your choice?

**Answer**:

**Question C.2.3 (1 mark)**: What is the forged message you found?

**Answer**:

**Task C.2 Source Code (1 mark)**

In [5]:
# TODO: Your code to achieve the subtask above. 



## Task C.3 (3 marks)

You are requested to employ **SHA3_256** to generate the digests for the message your chose in Task C.2 and the identified forged message, and then compare the two digests to check if they are equal to each other. Further, you are requested to calculate the percentage of difference between the two digests at the bit level, to observe how much avalanche effect can be obtained in modern cryptographic hash functions. How to use the SHA-3 family to generate hash values can be found here: https://cryptography.io/en/latest/hazmat/primitives/cryptographic-hashes/#sha-3-family

**Question C.3.1 (0.5 mark)**: What are the digests for the two messages?

**Answer**:

**Question C.3.2 (1 mark)**: What is the percentage of difference between the two digests at the bit level?

**Answer**:

**Question C.3.3 (0.5 mark)**: Which hash function has a stronger avalanche effect, SHA512 or TruncatedMD5? Why?

**Answer**:

**Task C.3 Source Code (1 mark)**

In [7]:
# TODO: Your code to generate digests and calculate the percentage of difference between the two digests at the bit level.

